# NLP Cheatsheet - Part 1: Basics

## 1. Text Representation

- Meaning: Converting text into numbers/vectors so ML/NLP models can understand and process it.
- Flow: Text -> Tokens -> Numerical IDs / Vectors
- Example: "FD interest rate is 7.5%" -> Tokens -> Numerical IDs/Vectors

## 2. Tokenization (Definition)

- Meaning: Splitting text into smaller units called tokens.
- Tokens can be: Words, Subwords, Characters
- Example: "FD interest rate 7.5%" -> ["FD", "interest", "rate", "7.5%"]

## 3. Types of Tokenization

### 3.1 Word-Level Tokenization

- Splits text into words using spaces/punctuation.
- Advantages: Simple, preserves word meaning.
- Example: "Fixed Deposit interest rate" -> ["Fixed", "Deposit", "interest", "rate"]
- Use Case: Bag of Words, TF-IDF
- Limitations:
  - x Huge vocabulary size.
  - x Cannot handle unknown/misspelled words.
  - x Cannot handle word variations.
- Example of limitation: "pay", "payment", "paid", "repayment" become separate tokens even though all are related to "payment" context.

### 3.2 Character-Level Tokenization

- Treats every character as a token.
- Advantages:
  - Handles spelling mistakes.
  - No unknown word problem.
- Use Case: Spelling correction, noisy text.
- Limitations:
  - x Very long sequences.
  - x Slower training.
  - x Characters lose semantic meaning.
- Example (banking narration noise): "SAL CR NEFT PAYTM LIMTED" - spelling may be inconsistent.
  - Reason: Since the model reads character-by-character, it becomes more robust to typos, missing characters, and noisy real-world text.
- Example:
  - "pay" -> ["p", "a", "y"]
  - "payment" -> ["p", "a", "y", "m", "e", "n", "t"]
  - "repayment" -> ["r", "e", "p", "a", "y", "m", "e", "n", "t"]
- Example (typo tolerance): "paymnt", "repaymnt", "paaymnt" - even with spelling mistakes, many characters are still similar, so the model can partially understand the word.

### 3.3 Subword Tokenization

- Sits in between Word-Level and Character-Level tokenization.
- Modern standard used in Transformer models.
- Breaks words into meaningful smaller units called subwords.
- Goal: Preserve meaning like word-level tokenization, while handling unknown words like character-level tokenization.
- Example:
  - "pay" -> ["pay"]
  - "payment" -> ["pay", "ment"]
  - "repayment" -> ["re", "pay", "ment"]
  - (Model understands all these words are related to the root concept "pay".)
- Advantages:
  - Handles rare/new words.
  - Smaller vocabulary.
  - Better multilingual handling.
- Most Common Algorithms: BPE, WordPiece, Unigram, SentencePiece

Subword Tokenization Flow (BPE Example):
- Step 1 - Start with characters: p a y m e n t
- Step 2 - Merge most frequent pairs: pa y m e n t
- Step 3 - Merge again: pay m e n t
- Step 4 - Final subwords: [pay] [ment]
- Similarly: "repayment" -> ["re", "pay", "ment"]

#### 3.3.1 Byte Pair Encoding (BPE)

- Type of Subword Tokenization.
- Used In: OpenAI GPT models, RoBERTa, CLIP
- Concept: Starts with characters and repeatedly merges the most frequent token pairs to form meaningful subword units.
  - "pay" -> ["pay"]
  - "payment" -> ["pay", "ment"]
  - "repayment" -> ["re", "pay", "ment"]
- How it learns: "pay" appears frequently -> merged as "pay". "re" appears frequently too. So instead of storing every full word separately, it stores reusable subwords.
- Advantages: Compact vocabulary. Handles unseen/new words efficiently.
- Example:
  - "prepayment" -> ["pre", "pay", "ment"]
  - "repayments" -> ["re", "pay", "ment", "s"]
- Limitation: Greedy merging may not always create linguistically meaningful tokens (splits based on frequency, not meaning).
  - "therapist" -> ["the", "rapist"]
  - "unhappy" -> ["un", "happy"]

#### 3.3.2 WordPiece Tokenization

- Used In: BERT
- Concept: Instead of storing every full word separately, BERT stores common subwords.
- Problem it solves: Storing every possible word (pay, payment, repayment, prepayment) makes vocabulary too large.
- Example:
  - "pay" -> ["pay"]
  - "payment" -> ["pay", "##ment"]
  - "repayment" -> ["re", "##pay", "##ment"]
- Meaning: "pay" = root word, "##ment" = continuation of previous token, "re" = "again". So BERT understands "payment" -> related to pay, and "repayment" -> pay again.
- Meaning of "##": Token continues the previous word.
  - "creditcard" -> ["credit", "##card"]
  - "overdues" -> ["over", "##due", "##s"]
  - "delayeddd" -> ["delay", "##ed", "##d"]
- Advantages: Handles unknown/new words. Smaller vocabulary. Robust to spelling mistakes.
- Example: "payment delayedd" -> even with the spelling mistake, BERT still understands meaning using known subwords like ["delay", "##ed", "##d"]

#### 3.3.3 SentencePiece

- Concept: Works directly on raw text and does NOT depend on spaces to identify words.
- Normal tokenizers first split sentence using spaces:
  - "FD not renewed" -> ["FD", "not", "renewed"]
- SentencePiece treats space itself as a token:
  - "FD not renewed" -> ["_FD", "_not", "_renew", "ed"]
- Meaning of "_": Represents the start of a new word/space.
- Why important: In many languages, spaces may not exist properly.
  - Examples: "FDnotrenewed", "salarydelayed", "paymentnotreceived"
  - SentencePiece can still learn meaningful splits like:
    - ["_FD", "not", "renew", "ed"]
    - ["_salary", "delay", "ed"]
- Main difference: Word tokenizer depends on spaces. SentencePiece learns directly from raw text, including spaces.

#### 3.3.4 Byte-Level BPE

- Used In: GPT-3, GPT-4
- Concept: Works at the byte level, so every symbol, emoji, spelling variation, or language can be processed.
- Example: Customer messages "payment" / "PAYMENT" / "payment@" / "paym&ent" - all can still be processed because byte-level encoding can represent every character/symbol.
- Why important: Traditional tokenizers may fail on special symbols or unseen characters, but Byte-Level BPE has NO Out-Of-Vocabulary (OOV) problem.

### 3.4 Quick Recap - Types of Tokenization

- Word-Level Tokenization: Splits text into complete words using spaces/punctuation.
- Character-Level Tokenization: Splits text into individual characters.
- Subword Tokenization: Splits words into meaningful smaller units (subwords) to handle unknown words efficiently.
- BPE (Byte Pair Encoding): Subword method that merges most frequent token pairs repeatedly.
- WordPiece Tokenization: Subword method used in BERT; builds words using subwords with "##" continuation.
- SentencePiece: Framework that tokenizes directly on raw text without depending on spaces.
- Byte-Level BPE: BPE variant that works on bytes; allows any character/emoji without OOV issues.